# LC 11 — Container With Most Water
**Day-37 | Two Pointers + Review**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Always move the shorter wall inward.
Moving the taller wall can only make things worse — the water
level is capped by the shorter side anyway.
</div>

## Official Problem Statement

You are given an integer array `height` of length `n`. There are
`n` vertical lines drawn such that the two endpoints of the `i`th
line are `(i, 0)` and `(i, height[i])`. Find two lines that
together with the x-axis form a container that holds the most
water. Return the maximum amount of water a container can store.

**Constraints:**
- `n == height.length`
- `2 <= n <= 10^5`
- `0 <= height[i] <= 10^4`

## What This Is Actually Asking

Pick two vertical bars from the array such that the rectangle
between them holds the most water. The width of the rectangle is
the distance between bar indices. The height is the shorter of
the two bars (water can only fill up to the shorter wall).
Maximize `min(h[i], h[j]) * (j - i)` over all valid pairs.

## Walk Through an Example by Hand

```
height = [1, 8, 6, 2, 5, 4, 8, 3, 7]

left=0, right=8: min(1,7)*8 =  8  move left  (h[0]=1 is shorter)
left=1, right=8: min(8,7)*7 = 49  move right (h[8]=7 is shorter)
left=1, right=7: min(8,3)*6 = 18  move right (h[7]=3 is shorter)
left=1, right=6: min(8,8)*5 = 40  tie => move either (move right)
left=1, right=5: min(8,4)*4 = 16  move right
left=1, right=4: min(8,5)*3 = 15  move right
left=1, right=3: min(8,2)*2 =  4  move right
left=1, right=2: min(8,6)*1 =  6  move right
left >= right, stop

Max water = 49
```

## The Picture

```
  8     |           |
  7     |           |       |
  6     |   |       |       |
  5     |   |   |   |       |
  4     |   |   | | |       |
  3     |   |   | | |   |   |
  2     |   | | | | |   |   |
  1   | |   | | | | |   |   |
  0   0 1 2 3 4 5 6 7 8
        L               R

water = min(h[L], h[R]) * (R - L)

Why move the shorter wall?
  If h[L] < h[R]:  water is capped at h[L]. Moving R inward
  reduces width AND keeps the same or smaller cap => worse.
  So move L inward — the only chance to improve.
```

## When To Use This Pattern

- When you need to maximize/minimize a product involving two
  indices and a min/max of their values, think **two pointers**.
- When moving one pointer can provably not improve the answer,
  think **greedy pointer advancement**.
- When an O(n²) pair scan is obvious but n is up to 10^5,
  think **can I reduce to O(n) with pointer logic?**
- When the result depends on `width * min(val_left, val_right)`,
  think **Container With Most Water pattern**.

## The Approach

Start with the widest possible container (`left=0, right=n-1`).
Compute the water and track the maximum. Then move the pointer
pointing to the shorter bar inward — this is the only hope of
finding a better container, since the width decreases regardless.
Repeat until the two pointers meet.

In [1]:
from typing import List

In [2]:
def test_harness(func):
    tests = [
        # (height, expected)
        ([1, 8, 6, 2, 5, 4, 8, 3, 7], 49),
        ([1, 1],                        1),
        ([4, 3, 2, 1, 4],              16),
        ([1, 2, 1],                     2),
        ([0, 0],                        0),   # edge: all zeros
    ]
    passed = 0
    for height, expected in tests:
        result = func(height)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"{status} | height={height} "
              f"| got={result}, expected={expected}")
    print(f"\n{passed}/{len(tests)} tests passed")

In [3]:
def max_area(nums: List[int]) -> int:
    """
    Two-pointer squeeze: always move the shorter wall.

    Args:
        height: List of non-negative integers representing bar heights.

    Returns:
        Maximum water that can be trapped between two bars.

    Time:  O(n)  — single pass, each pointer moves at most n times
    Space: O(1)  — only a few integer variables
    """
    l, r = 0 , len(nums)-1
    res = 0
    while l < r:
        res = max(res, (r-l) * min (nums[l], nums[r]))
        if nums[l] < nums[r]:
            l+=1
        else:
            r-=1
    return res
# Quick debug — run this cell while building
print(max_area([1,8,6,2,5,4,8,3,7]))  # 49
print(max_area([1,1]))                 # 1
print(max_area([4,3,2,1,4]))           # 16
print(max_area([1,0,0,0,1]))           # 4
test_harness(max_area)

49
1
16
4
PASSED | height=[1, 8, 6, 2, 5, 4, 8, 3, 7] | got=49, expected=49
PASSED | height=[1, 1] | got=1, expected=1
PASSED | height=[4, 3, 2, 1, 4] | got=16, expected=16
PASSED | height=[1, 2, 1] | got=2, expected=2
PASSED | height=[0, 0] | got=0, expected=0

5/5 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(max_area)

## Complexity

| Approach      | Time   | Space | Notes                              |
|---------------|--------|-------|------------------------------------|
| Brute force   | O(n²)  | O(1)  | Check every pair of bars           |
| Two pointers  | O(n)   | O(1)  | Optimal; greedy shorter-wall move  |

## Real World Connection

This greedy "move the weaker side" logic appears in network
bandwidth optimization: a connection's throughput is capped by
the slowest link, so you upgrade the weakest node first. In data
engineering, when joining two data streams the throughput is
limited by the slower producer — a concept directly mirroring
the shorter-wall principle. At AWS, designing S3 read pipelines
with multiple parallel readers uses similar max-flow reasoning.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra